# ዘር · Zer — QLoRA fine-tuning on a free GPU

Fine-tune an open model (Qwen2.5 by default) on **Zer's Amharic dataset**—built
straight from the repo—using 4-bit QLoRA.

**Works on:** Google Colab (free T4) and Kaggle (T4/P100).

> **Use your free GPU within the provider's rules.** Don't run multiple
> accounts or parallel sessions to dodge session/time limits — that breaks
> their Terms and can get you banned. 1.5 B fits any T4; try 3 B for better
> quality (batch 1–2). 7 B needs a bigger GPU than the free tier.

**After training:** download `zer-lora.zip` (or push the adapter to the Hugging
Face Hub) and serve it with vLLM / llama.cpp behind `LLM_BASE_URL`.

In [ ]:
# 1) Confirm you actually have a GPU
!nvidia-smi || echo 'No GPU — switch the Colab runtime to GPU (Runtime ▸ Change runtime type ▸ T4 GPU)'

In [ ]:
# 2) Get the code
%cd /content 2>/dev/null || %cd /kaggle/working
import os
if not os.path.isdir('amharic-nlp-chatbot'):
    !git clone --depth 1 https://github.com/ZebraCodeX/amharic-nlp-chatbot.git
%cd amharic-nlp-chatbot

In [ ]:
# 3) Install the training stack (torch is already present on Colab/Kaggle)
!pip install -q -U 'transformers<5' peft accelerate datasets bitsandbytes sentencepiece safetensors
import torch, transformers, peft
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, '| peft', peft.__version__)

In [ ]:
# 4) Build the Amharic SFT dataset from the repo's own data
!python training/build_dataset.py
!wc -l training/data/amharic_sft.jsonl
!head -c 400 training/data/amharic_sft.jsonl

### 5) Train

Default is **Qwen2.5-1.5B** (fits any T4). For better quality switch `MODEL` to
`Qwen/Qwen2.5-3B-Instruct` and lower the batch (`BATCH=1`, `GA=16`, `MAXLEN=1024`).
This takes ~15–40 min on a T4 for the seed set.

In [ ]:
MODEL  = 'Qwen/Qwen2.5-1.5B-Instruct'   # or 'Qwen/Qwen2.5-3B-Instruct'
OUT    = 'training/out/zer-lora'
EPOCHS, BATCH, GA, MAXLEN, LR = 3, 2, 8, 1024, 2e-4

!python training/train_qlora.py \
  --model {MODEL} --dataset training/data/amharic_sft.jsonl --out {OUT} \
  --epochs {EPOCHS} --batch {BATCH} --grad-accum {GA} --max-len {MAXLEN} --lr {LR}

In [ ]:
# 6) Merge the LoRA adapter into a standalone model (small; keeps the repo light)
!python training/merge_adapter.py --base {MODEL} --adapter {OUT} --out training/out/zer-merged

In [ ]:
# 7) Sanity check: does it answer in the same language as the prompt?
!python training/eval.py --model training/out/zer-merged --limit 5

In [ ]:
# 8) Save the adapter (tiny) and download it — Colab and Kaggle both handled
import os, shutil, glob
shutil.make_archive('zer-lora', 'zip', OUT)
print('adapter size (MB):', round(os.path.getsize('zer-lora.zip') / 1e6, 2))
print('merged model files:', glob.glob('training/out/zer-merged/*')[:4], '…')
try:
    from google.colab import files  # Colab only
    files.download('zer-lora.zip')
except Exception:
    print('Kaggle: your outputs in /kaggle/working are saved automatically.')

### 9) Optional — publish to the Hugging Face Hub

Set `HF_TOKEN` (or paste it) and push the adapter so any GPU host can pull it.

In [ ]:
# import os
# os.environ['HF_TOKEN'] = 'hf_...'   # a token with write access
# REPO = 'your-username/zer-qwen-lora'
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo(REPO, exist_ok=True, private=False)
# api.upload_folder(folder_path=OUT, repo_id=REPO)
# print('pushed → https://huggingface.co/' + REPO)

### 10) Serve it and connect the app

On a GPU host with your adapter/model:

```bash
pip install vllm
vllm serve training/out/zer-merged --served-model-name zer --port 8000
```

Then point the deployed app at it (no redeploy needed):

```bash
flyctl secrets set LLM_BASE_URL=https://YOUR-GPU-HOST/v1 LLM_MODEL=zer LLM_API_KEY=optional
```

Zer automatically prefers the LLM for creative/open questions while keeping the
offline Amharic brain (code, knowledge, translation) as the fallback.